In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml down minio

In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml up -d minio


#### Delete Existing Delta Table

In [ ]:
%%bash

## Delete Existing Delta Table
aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/delta_table --recursive


#### Create Deltatable

In [1]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", TimestampType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

df = spark.read.format("csv").option("header", True).schema(schema).load("s3a://datasets/people-data/peoples.csv")
df.printSchema()

#
display(df)

root
 |-- id: integer (nullable = true)
 |-- firstName: string (nullable = true)
 |-- middleName: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthDate: timestamp (nullable = true)
 |-- ssn: string (nullable = true)
 |-- salary: integer (nullable = true)

+---+----------+----------+-------------+------+-------------------+-----------+------+
|id |firstName |middleName|lastName     |gender|birthDate          |ssn        |salary|
+---+----------+----------+-------------+------+-------------------+-----------+------+
|1  |Pennie    |Carry     |Hirschmann   |F     |1955-07-02 09:30:00|981-43-9345|56172 |
|2  |An        |Amira     |Cowper       |F     |1992-02-08 10:30:00|978-97-8086|40203 |
|3  |Quyen     |Marlen    |Dome         |F     |1970-10-11 09:30:00|957-57-8246|53417 |
|4  |Coralie   |Antonina  |Marshal      |F     |1990-04-11 09:30:00|963-39-4885|94727 |
|5  |Terrie    |Wava      |Bonar        |F     |1980-01-16 10:30

#### Create Delta table

In [ ]:
# Save as delta table into Minio S3
df.write.format('delta').save('s3a://warehouse/default/deltalake/delta_table')

# Create the table if it does not exist. Otherwise, replace the existing table.
#df.writeTo("spark_catalog.deltalake.delta_table").createOrReplace()

# If you know the table does not already exist, you can call this instead:
#df.write.saveAsTable("spark_catalog.deltalake.delta_table")

##### Write into existing table

In [ ]:
df.write.format("delta").mode("append").saveAsTable("spark_catalog.deltalake.delta_table")

#
# df.write.format("delta").mode("append").save("s3a://warehouse/default/deltalake/delta_table")

#
# df.write.insertInto("spark_catalog.deltalake.delta_table")

#### Read a Deltatable

In [4]:
delta_table = spark.read.format('delta').load("s3a://warehouse/default/deltalake/delta_table")
display(delta_table)

+---+----------+----------+-------------+------+-------------------+-----------+------+
|id |firstName |middleName|lastName     |gender|birthDate          |ssn        |salary|
+---+----------+----------+-------------+------+-------------------+-----------+------+
|1  |Pennie    |Carry     |Hirschmann   |F     |1955-07-02 09:30:00|981-43-9345|56172 |
|2  |An        |Amira     |Cowper       |F     |1992-02-08 10:30:00|978-97-8086|40203 |
|3  |Quyen     |Marlen    |Dome         |F     |1970-10-11 09:30:00|957-57-8246|53417 |
|4  |Coralie   |Antonina  |Marshal      |F     |1990-04-11 09:30:00|963-39-4885|94727 |
|5  |Terrie    |Wava      |Bonar        |F     |1980-01-16 10:30:00|964-49-8051|79908 |
|6  |Chassidy  |Concepcion|Bourthouloume|F     |1990-11-24 10:30:00|954-59-9172|64652 |
|7  |Geri      |Tambra    |Mosby        |F     |1970-12-19 10:30:00|968-16-4020|38195 |
|8  |Patria    |Nancy     |Arstall      |F     |1985-01-02 10:30:00|984-76-3770|102053|
|9  |Terese    |Alfredia  |Tocqu

In [ ]:
%load_ext sql

In [6]:
%sql spark

In [7]:
%%sql

DESCRIBE HISTORY delta.`s3a://warehouse/default/deltalake/delta_table`

Running query in 'SparkSession'

2 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
1,2026-04-25 08:53:35,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,0,Serializable,True,"{'numOutputRows': '1000', 'numOutputBytes': '47305', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-04-25 08:31:45,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,True,{},None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [8]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 's3a://warehouse/default/deltalake/delta_table')
display(deltaTable.history())

+-------+-------------------+------+--------+------------+----------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation   |operationParameters                                                                           |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                               |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+------------+----------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------+------------+-----------------------------------+
|1  

In [ ]:
%%sql

INSERT INTO  delta.`s3a://warehouse/default/deltalake/delta_table`
SELECT * FROM PARQUET.`s3a://warehouse/default/deltalake/delta_table`;

In [9]:
%%sql

SELECT MIN(salary), MAX(salary), COUNT(id)
FROM delta.`s3a://warehouse/default/deltalake/delta_table`

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3
-6523,134393,1000


In [10]:
%%sql 

UPDATE delta.`s3a://warehouse/default/deltalake/delta_table`
SET salary = 102053
WHERE salary = 10205;

Running query in 'SparkSession'

1 rows affected.

Field 1
0


In [11]:
%%sql

DESCRIBE HISTORY delta.`s3a://warehouse/default/deltalake/delta_table`

Running query in 'SparkSession'

2 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
1,2026-04-25 08:53:35,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,0,Serializable,True,"{'numOutputRows': '1000', 'numOutputBytes': '47305', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-04-25 08:31:45,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,True,{},None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [12]:
%%sql

DESCRIBE DETAIL spark_catalog.deltalake.delta_table
--DESCRIBE DETAIL delta.`s3a://warehouse/default/deltalake/delta_table`

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
delta,7a1f3801-805d-40fc-9547-e9270e0593fa,spark_catalog.deltalake.delta_table,None,s3a://warehouse/default/deltalake/delta_table,2026-04-25 08:31:44.915000,2026-04-25 08:53:35,[],[],1,47305,{},1,2,"['appendOnly', 'invariants']"


In [13]:
%%sql 

SELECT *
FROM delta.`s3a://warehouse/default/deltalake/delta_table`
WHERE id = 1

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8
1,Pennie,Carry,Hirschmann,F,1955-07-02 09:30:00,981-43-9345,56172


In [ ]:
%%sql 

DELETE
FROM delta.`s3a://warehouse/default/deltalake/delta_table`
WHERE id = 1

In [ ]:
%%sql 

SELECT *
FROM delta.`s3a://warehouse/default/deltalake/delta_table`
WHERE id = 1

In [14]:
%%sql

DESCRIBE HISTORY delta.`s3a://warehouse/default/deltalake/delta_table`

Running query in 'SparkSession'

2 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
1,2026-04-25 08:53:35,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,0,Serializable,True,"{'numOutputRows': '1000', 'numOutputBytes': '47305', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-04-25 08:31:45,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,True,{},None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [ ]:
for i in range(5):
    spark.sql(
        """
        INSERT INTO delta.`s3a://warehouse/default/deltalake/delta_table`
        SELECT * FROM PARQUET.`s3a://warehouse/default/deltalake/delta_table`;
        """
    )
    print(f"{i}th INSERT completed ")

In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://warehouse/default/deltalake/delta_table`

#### Python Create Deltatable

In [ ]:
spark.sql("""

CREATE OR REPLACE TABLE delta.`s3a://warehouse/default/deltalake/delta_table` (
  id INT,
  firstName STRING,
  middleName STRING,
  lastName STRING,
  gender STRING,
  birthDate TIMESTAMP,
  ssn STRING,
  salary INT
) USING DELTA
          
""")

In [ ]:
from delta import *

DeltaTable.createIfNotExists(spark) \
    .tableName("delta_table") \
    .addColumn("id", "INT") \
    .addColumn("firstName", "STRING") \
    .addColumn("middleName", "STRING") \
    .addColumn("lastName", "STRING", comment = "surname") \
    .addColumn("gender", "STRING") \
    .addColumn("birthDate", "TIMESTAMP") \
    .addColumn("ssn", "STRING") \
    .addColumn("salary", "INT") \
    .location("s3a://warehouse/default/deltalake/delta_table") \
    .execute()

In [ ]:
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, 's3a://warehouse/default/deltalake/delta_table')
deltaTable.toDF().show()


In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://warehouse/deltalake/deltalake/delta_table`

In [ ]:
%%sql 

CREATE TABLE IF NOT EXISTS spark_catalog.deltalake.people12m (
    id INT,
    firstName STRING,
    middleName STRING,
    lastName STRING,
    gender STRING,
    birthDate TIMESTAMP,
    ssn STRING,
    salary INT
)
USING DELTA
--LOCATION 's3a://warehouse/deltalake/deltalake/people12m'

In [ ]:
%%sql 

DESCRIBE TABLE FORMATTED spark_catalog.deltalake.people12m

In [ ]:
%%sql 

DESCRIBE TABLE EXTENDED spark_catalog.deltalake.people12m;

In [ ]:
%%sql 

DESCRIBE HISTORY spark_catalog.deltalake.people12m;

#### Upsert to a Deltatable

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from datetime import date

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", DateType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

data = [
  (9999998, 'Billy', 'Tommie', 'Luppitt', 'M', date.fromisoformat('1992-09-17'), '953-38-9452', 55250),
  (9999999, 'Elias', 'Cyril', 'Leadbetter', 'M', date.fromisoformat('1984-05-22'), '906-51-2137', 48500),
  (10000000, 'Joshua', 'Chas', 'Broggio', 'M', date.fromisoformat('1968-07-22'), '988-61-6247', 90000),
  (20000001, 'John', '', 'Doe', 'M', date.fromisoformat('1978-01-14'), '345-67-8901', 55500),
  (20000002, 'Mary', '', 'Smith', 'F', date.fromisoformat('1982-10-29'), '456-78-9012', 98250),
  (20000003, 'Jane', '', 'Doe', 'F', date.fromisoformat('1981-06-25'), '567-89-0123', 89900)
]

delta_table_updates = spark.createDataFrame(data, schema)
delta_table_updates.createOrReplaceTempView("spark_catalog.deltalake.delta_table_updates")

# ...

from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, 's3a://warehouse/default/deltalake/delta_table')

(deltaTable.alias("delta_table")
  .merge(
    delta_table_updates.alias("delta_table_updates"),
    "delta_table.id = delta_table_updates.id"
  )
  .whenMatchedUpdateAll()
  .whenNotMatchedInsertAll()
  .execute()
)

#### Read a Deltatable

In [ ]:
df = spark.read.format('delta').load("s3a://warehouse/default/deltalake/delta_table")
df_filtered = df.filter(df["id"] >= 9999998)
display(df_filtered)

In [ ]:
from pyspark.sql.functions import col
df = spark.read.option("versionAsOf", "1").table("delta.`s3a://warehouse/default/deltalake/delta_table`")
display(df.filter(col("id") == 1))

#### Write to a Deltatable

In [ ]:
# df.write.mode("append").saveAsTable("spark_catalog.deltalake.delta_table")

# Save as delta table
df.write.format('delta').mode('append').save('s3a://warehouse/default/deltalake/delta_table')

In [ ]:
# df.write.mode("overwrite").saveAsTable("spark_catalog.deltalake.delta_table")

# Save as delta table
df.write.format('delta').mode('overwrite').save('s3a://warehouse/default/deltalake/delta_table')

#### Update Deltatable Rows

In [ ]:
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, 's3a://warehouse/default/deltalake/delta_table')

# Declare the predicate by using a SQL-formatted string.
deltaTable.update(
  condition = "gender = 'F'",
  set = { "gender": "'Female'" }
)

# Declare the predicate by using Spark SQL functions.
deltaTable.update(
  condition = col('id') % lit(2) == 0,
  set = { "gender": "'Male'" }
)

In [ ]:
df = spark.read.format('delta').load("s3a://warehouse/default/deltalake/delta_table")
# df.limit(20).show()

df_filtered = df.filter((col("gender") == "Male"))
display(df_filtered)

#### Delete Rows 

In [ ]:
from delta.tables import *
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, 's3a://warehouse/default/deltalake/delta_table')

# Declare the predicate by using a SQL-formatted string.
deltaTable.delete("birthDate < '1955-01-01'")

# Declare the predicate by using Spark SQL functions.
deltaTable.delete(col('birthDate') < '1960-01-01')

#### Display table history

In [ ]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 's3a://warehouse/default/deltalake/delta_table')
display(deltaTable.history())

#### overwrite

In [ ]:
# Save as delta table
df.write.format('delta').mode('overwrite').save('s3a://warehouse/default/deltalake/delta_table')

#### Time Travle

In [ ]:
# Read version 1
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 's3a://warehouse/default/deltalake/delta_table')
deltaHistory = deltaTable.history()

display(deltaHistory.where("version == 0"))
# Or:
#display(deltaHistory.where("timestamp == '2024-05-15T22:43:15.000+00:00'"))

In [ ]:
df = spark.read.format('delta').option('versionAsOf', 1).load("s3a://warehouse/default/deltalake/delta_table")
# Or: 2025-10-14 18:39:41
#df = spark.read.format('delta').option('timestampAsOf', '2025-10-14T18:45:03.000+00:00').load("/deltalake/delta_table")

display(df)

#### Display table history

In [ ]:
deltaTable = DeltaTable.forPath(spark, "s3a://warehouse/default/deltalake/delta_table")
print("######## Describe history for the table ######")
deltaTable.history().show()

#### Vacuum

In [ ]:
deltaTable = DeltaTable.forPath(spark, "s3a://warehouse/default/deltalake/delta_table")
print("######## Vacuum the table ########")
deltaTable.vacuum()

#### Describe details for the table

In [ ]:
print("######## Describe details for the table ######")
deltaTable.detail().show()

#### Generating manifest 

In [ ]:
# Generate manifest
print("######## Generating manifest ######")
deltaTable.generate("SYMLINK_FORMAT_MANIFEST")

In [ ]:
# SQL Vacuum
print("####### SQL Vacuum #######")
spark.sql("VACUUM '%s' RETAIN 169 HOURS" % ("s3a://warehouse/default/deltalake/delta_table")).collect()

In [ ]:
# SQL describe history
print("####### SQL Describe History ########")
print(spark.sql("DESCRIBE HISTORY delta.`%s`" % ("s3a://warehouse/default/deltalake/delta_table")).collect())

In [ ]:
import shutil

# cleanup
shutil.rmtree("/warehouse/default/deltalake/delta_table")

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/delta_table --recursive

#### Optimize a table
After you have performed multiple changes to a table, you might have a lot of small files. To improve the speed of read queries, you can use the optimize operation to collapse small files into larger ones:

In [ ]:
from delta.tables import *

#deltaTable = DeltaTable.forName(spark, "spark_catalog.deltalake.delta_table")
deltaTable = DeltaTable.forPath(spark, "s3a://warehouse/default/deltalake/delta_table")
deltaTable.optimize().executeCompaction()

#### Z-order by columns
To improve read performance further, you can collocate related information in the same set of files by z-ordering. Delta Lake data-skipping algorithms use this collocation to dramatically reduce the amount of data that needs to be read. To z-order data, you specify the columns to order on in the z-order by operation. For example, to collocate by gender, run:

In [ ]:
from delta.tables import *

#
# deltaTable = DeltaTable.forName(spark, "spark_catalog.deltalake.delta_table")

deltaTable = DeltaTable.forPath(spark, "s3a://warehouse/default/deltalake/delta_table")
deltaTable.optimize().executeZOrderBy("gender")

#### Clean up snapshots with VACUUM
Delta Lake provides snapshot isolation for reads, which means that it is safe to run an optimize operation even while other users or jobs are querying the table. Eventually however, you should clean up old snapshots. You can do this by running the vacuum operation:

In [ ]:
from delta.tables import *

#
# deltaTable = DeltaTable.forName(spark, "spark_catalog.deltalake.delta_table")

deltaTable = DeltaTable.forPath(spark, "s3a://warehouse/default/deltalake/delta_table")
deltaTable.vacuum()

#### How do I find the last commit's version in the Spark session?

In [ ]:
spark.conf.get("spark.databricks.delta.lastCommitVersionInSession")